<a href="https://colab.research.google.com/github/Pranayshukla0610/Financial-Analyst/blob/main/Power_BI_Style_Financial_Dashboard_using_Python_%26_Bokeh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [188]:
import yfinance as yf
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from math import pi

from bokeh.io import output_notebook, show
from bokeh.io import curdoc
from bokeh.layouts import row, column
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    Div,
    Select
)

In [189]:
output_notebook()

In [190]:
BACKGROUND = "#0B1120"
CARD = "#1E293B"
TEXT = "#F8FAFC"
GRID = "#334155"

WIDTH = 650
HEIGHT = 400

In [191]:
companies = {
    "Netflix":"NFLX",
    "Tesla":"TSLA",
    "NVIDIA":"NVDA",
    "Amazon":"AMZN",
    "Meta":"META"
}

company_selector = Select(
    title="Select Company",
    value="Netflix",
    options=list(companies.keys())
)

In [192]:
company = company_selector.value

ticker = companies[company]

print(ticker)

NFLX


In [193]:
ticker = companies['Netflix']

data = yf.download(
    ticker,
    period = "2y"
)

/tmp/ipykernel_33821/4158580776.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed


In [194]:
data.head()

Price,Close,High,Low,Open,Volume
Ticker,NFLX,NFLX,NFLX,NFLX,NFLX
Date,,,,,
2024-05-09,61.209000,61.571999,60.575001,61.439999,20654000
2024-05-10,61.087002,62.397999,60.506001,61.900002,26536000
2024-05-13,61.659000,61.821999,60.682999,61.430000,20862000
2024-05-14,61.366001,62.147999,60.840000,61.516998,27924000
2024-05-15,61.352001,62.410000,60.910000,61.856998,56706000


In [195]:
data.reset_index(inplace=True)

In [196]:
data.columns = data.columns.get_level_values(0)

In [197]:
data['Returns'] = data['Close'].pct_change()

In [198]:
#MOVING AVERAGES

data['MA20'] = data['Close'].rolling(20).mean()
data['MA50'] = data['Close'].rolling(50).mean()

In [199]:
data['Volatility'] = data['Returns'].rolling(20).std()

In [200]:
data['Upper'] = (
    data['MA20'] + 2 * data['Volatility']
)

data['Lower'] = (
    data['MA20'] - 2 * data['Volatility']
)

In [201]:
investment = 10000

shares = (
    investment / data['Close'].iloc[0]
)

data['Portfolio'] = (
    shares * data['Close']
)

In [202]:
data.dropna(inplace=True)

In [203]:
source = ColumnDataSource(data)

In [204]:
latest_price = round(
    data['Close'].iloc[-1],
    2
)

In [205]:
risk_free_rate = 0.02

sharpe_ratio = (
    (
        data['Returns'].mean() * 252
        - risk_free_rate
    )
    /
    (
        data['Returns'].std()
        * np.sqrt(252)
    )
)

In [206]:
inc = data['Close'] > data['Open']
dec = data['Close'] < data['Open']

In [207]:
latest_price = round(
    data['Close'].iloc[-1],
    2
)

daily_return = round(
    data['Close'].iloc[-1] * 100,
    2
)

volatility = round(
    data['Volatility'].mean() * 100,
    2
)

portfolio_value = round(
    data['Portfolio'].iloc[-1],
    2
)

profit = round(
    portfolio_value - investment,
    2
)

In [208]:
def create_kpi(title, value, color):

    return Div(text=f"""

    <div style="
    background:{color};
    padding:20px;
    border-radius:18px;
    text-align:center;
    width:220px;
    height:120px;
    box-shadow:0px 4px 15px rgba(0,0,0,0.5);
    ">

    <h2 style="
    color:white;
    font-family:Arial;
    ">
    {title}
    </h2>

    <h1 style="
    color:white;
    ">
    {value}
    </h1>

    </div>
    """)

In [209]:
kpi = create_kpi(
    "Live Price",
    f"${latest_price}",
    "#2563EB"
)

kpi2 = create_kpi(
    "Daily Return",
    f"{daily_return}%",
    "#059669"
)

kpi3 = create_kpi(
    "Volatility",
    f"{volatility}%",
    "DC2626"
)

kpi4 = create_kpi(
    "Portfolio Value",
    f"${portfolio_value}",
    "#7C3AED"
)

kpi5 = create_kpi (
    "Profit/Loss",
    f"${profit}",
    "#EA580C"
)

kpi6 = create_kpi (
    "Sharpe Ratio",
    sharpe_ratio,
    "#0891B2"
)

In [210]:
p1 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Candlestick Analysis",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [211]:
p1.segment(
    data['Date'],
    data['High'],
    data['Date'],
    data['Low'],
    color="white"
)

GlyphRenderer(id='p2727', ...)

In [212]:
w = 12 * 60 * 60 * 1000

In [213]:
p1.vbar(
    data.loc[inc,'Date'],
    w,
    data.loc[inc,'Open'],
    data.loc[inc,'Close'],
    fill_color="#10B981",
    line_color="#10B981"
)

GlyphRenderer(id='p2738', ...)

In [214]:
p1.vbar(
    data.loc[dec,'Date'],
    w,
    data.loc[dec,'Open'],

    data.loc[dec,'Close'],
    fill_color="#EF4444",
    line_color="#EF4444"
)

GlyphRenderer(id='p2749', ...)

In [215]:
p2 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Moving Average Analytics",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [216]:
p2.line(
    data['Date'],
    data['Close'],
    line_width=2,
    color="white",
    legend_label="Close Price"
)

GlyphRenderer(id='p2809', ...)

In [217]:
p2.line(
    data['Date'],
    data['MA20'],
    line_width=3,
    color="#3B82F6",
    legend_label="MA20"
)

GlyphRenderer(id='p2822', ...)

In [218]:
p2.line(
    data['Date'],
    data['MA50'],
    line_width=3,
    color="#F59E0B",
    legend_label="MA50"
)

GlyphRenderer(id='p2834', ...)

In [219]:
p3 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Volume Analysis",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [220]:
p3.vbar(
    x=data['Date'],
    top=data['Volume'],
    width=w,
    color="#06B6D4"
)

GlyphRenderer(id='p2895', ...)

In [221]:
p4 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Risk & Volatility",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [222]:
p4.line(
    data['Date'],
    data['Volatility'],
    line_width=3,
    color="#EF4444"
)

GlyphRenderer(id='p2955', ...)

In [223]:
hist, edges = np.histogram(
    data['Returns'].dropna(),
    bins=40
)

In [224]:
p5 = figure(
    width=WIDTH,
    height=HEIGHT,
    title="Return Distribution",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [225]:
p5.quad(
    top=hist,
    bottom=0,
    left=edges[:-1],
    right=edges[1:],
    fill_color="#8B5CF6",
    line_color="white"
)

GlyphRenderer(id='p3001', ...)

In [226]:
p6 = figure(
    x_axis_type="datetime",
    width=WIDTH,
    height=HEIGHT,
    title="Portfolio Growth",
    background_fill_color=CARD,
    border_fill_color=BACKGROUND
)

In [227]:
p6.line(
    data['Date'],
    data['Portfolio'],
    line_width=4,
    color="#22C55E"
)

GlyphRenderer(id='p3061', ...)

In [228]:
hover = HoverTool(
    tooltips=[
        ("Date", "@Date{%F}"),
        ("Open", "@Open"),
        ("Close", "@Close"),
        ("High", "@High"),
        ("Low", "@Low")
    ],

    formatters={
        '@Date': 'datetime'
    }
)

In [229]:
p1.add_tools(hover)

In [230]:
plots = [p1,p2,p3,p4,p5,p6]

In [231]:
for p in plots:

    p.title.text_color = TEXT

    p.xaxis.major_label_text_color = TEXT
    p.yaxis.major_label_text_color = TEXT

    p.xaxis.axis_label_text_color = TEXT
    p.yaxis.axis_label_text_color = TEXT

    p.grid.grid_line_color = GRID

In [232]:
title = Div(text="""

<div style="
background:linear-gradient(
90deg,
#111827,
#1D4ED8
);

padding:25px;
border-radius:18px;
box-shadow:0px 4px 15px rgba(0,0,0,0.5);
">

<h1 style="
color:white;
text-align:center;
font-size:40px;
font-family:Arial;
">

NETFLIX FINANCIAL ANALYTICS DASHBOARD

</h1>

</div>

""")

In [233]:
dashboard = column(

    title,

    row(
        kpi,
        kpi2,
        kpi3
    ),

    row(
        kpi4,
        kpi5,
        kpi6
    ),

    row(
        company_selector
    ),

    row(
        p1,
        p2
    ),

    row(
        p3,
        p4
    ),

    row(
        p5,
        p6
    )
)

In [234]:
show(dashboard)